# Wrap the RAG chain in a custom pyfunc model with pre and post processing, log it via MLflow, and register it to Unity Catalog with a champion alias

The chain works in a notebook but the audit team will not approve a "just-a-notebook" deployment. Every production agent needs a UC-registered model version so they can trace any inference to a specific commit. You have one session to wrap, sign, log, register, alias, load-back, and run a smoke test.

The hands-on work:

- Write a subclass of `mlflow.pyfunc.PythonModel` whose `predict()` runs pre-processing (lowercase, strip, length-check), an inline retrieve-and-answer chain, and post-processing (trim plus citation footer).
- Log the pyfunc with an explicit signature inferred from an example input and output.
- Register it to Unity Catalog at the full three-level name `workspace.default.labrun_genai_pyfunc.rag_agent` and set the `champion` alias.
- Load the model back via the alias URI and prove a round-trip inference works.

Estimated time: 70 minutes. Cost: $0.00 to $0.02 (MLflow logging and UC registration are free; the round-trip inference is fractions of a cent).


In [ ]:
# NBVAL_SKIP
# Pinned dependencies. langchain + databricks-langchain are listed because
# pyfunc inference imports them at load_context time.

!pip install --quiet labrun-checks==0.7.0 databricks-sdk==0.40.0 mlflow==2.20.0 langchain==0.3.7 databricks-langchain==0.1.1


In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. UC identifiers (schema, table, volume) use
# snake_case under the labrun_ prefix because Unity Catalog identifiers prefer
# snake_case (hyphens force backtick-quoting everywhere downstream).

import atexit
import getpass
import json
import re
import sys
import time

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupAdapter,
    CleanupResource,
)

LAB_ID = "mlflow-pyfunc-rag-to-unity-catalog"
LAB_TAG_KEY = "labrun_lab_slug"
LAB_TAG_VALUE = LAB_ID  # must equal cert YAML labs[].slug exactly

CATALOG = "workspace"
PARENT_SCHEMA = "default"
LAB_SCHEMA = "labrun_genai_pyfunc"
LAB_SCHEMA_FQN = f"{CATALOG}.{PARENT_SCHEMA}.{LAB_SCHEMA}"

MODEL_SCHEMA_FQN = LAB_SCHEMA_FQN  # workspace.default.labrun_genai_pyfunc
REGISTERED_MODEL_NAME = f"{LAB_SCHEMA_FQN}.rag_agent"
ROUNDTRIP_TABLE_FQN = f"{LAB_SCHEMA_FQN}.roundtrip_results"
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
EMBEDDING_ENDPOINT = "databricks-gte-large-en"
MLFLOW_EXPERIMENT_NAME = None  # resolved post-register()
CHAMPION_ALIAS = "champion"


In [ ]:
# NBVAL_SKIP
# Register the session, validate Databricks credentials via the SDK, and
# resolve the Starter SQL warehouse. This cell must succeed before the
# manifest cell runs anything.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
databricks_host = getpass.getpass("Databricks workspace URL (https://...azuredatabricks.net or .cloud.databricks.com): ").strip()
databricks_token = getpass.getpass("Databricks personal access token (PAT): ").strip()

if not databricks_host or not databricks_token:
    print("Workspace URL and PAT are both required. Re-run this cell.")
    raise SystemExit(1)
if not databricks_host.startswith("https://"):
    databricks_host = f"https://{databricks_host}"

w = WorkspaceClient(host=databricks_host, token=databricks_token)

try:
    me = w.current_user.me()
except Exception as exc:
    print("Databricks credentials are missing or expired. CurrentUser.me() failed:")
    print(f"  {exc}")
    print()
    print("Refresh your workspace URL and PAT, then re-run this cell.")
    raise SystemExit(1)

CALLER_USER = me.user_name or (me.emails[0].value if me.emails else "unknown")
print(f"Credentials validated. Workspace user: {CALLER_USER}")

warehouses = list(w.warehouses.list())
if not warehouses:
    print("No SQL warehouses found in this workspace. Free Edition ships with")
    print("a Starter warehouse by default; if it has been deleted, recreate it")
    print("from the SQL > Warehouses page before re-running this cell.")
    raise SystemExit(1)
WAREHOUSE = warehouses[0]
WAREHOUSE_ID = WAREHOUSE.id
print(f"SQL warehouse resolved: {WAREHOUSE.name} ({WAREHOUSE_ID})")

DATABRICKS_CREDENTIALS = {
    "host": databricks_host,
    "token": databricks_token,
    "warehouse_id": WAREHOUSE_ID,
}


def run_sql(statement, warehouse_id=None, wait_seconds=180):
    """Submit SQL to the warehouse and return {columns, rows}."""
    wid = warehouse_id or WAREHOUSE_ID
    resp = w.statement_execution.execute_statement(
        warehouse_id=wid, statement=statement, wait_timeout="50s"
    )
    statement_id = resp.statement_id
    deadline = time.time() + wait_seconds
    while True:
        state = resp.status.state if resp.status else None
        if state == StatementState.SUCCEEDED:
            break
        if state in (StatementState.FAILED, StatementState.CANCELED, StatementState.CLOSED):
            err = resp.status.error.message if (resp.status and resp.status.error) else "no error message"
            raise RuntimeError(f"SQL failed ({state}): {err}\n  Statement: {statement}")
        if time.time() > deadline:
            raise TimeoutError(
                f"SQL did not finish in {wait_seconds}s. Last state: {state}. "
                f"The Starter warehouse may still be waking up; re-run the cell."
            )
        time.sleep(2)
        resp = w.statement_execution.get_statement(statement_id)
    columns = []
    if resp.manifest and resp.manifest.schema and resp.manifest.schema.columns:
        columns = [c.name for c in resp.manifest.schema.columns]
    rows = []
    if resp.result and resp.result.data_array:
        rows = list(resp.result.data_array)
    return {"columns": columns, "rows": rows}


register(session_token=session_token, lab_id=LAB_ID, credentials=DATABRICKS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")


In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan.
# Manifest is module-level and in reverse-creation order. Per
# RESOURCE-SAFETY-SPEC Rule 4 the orphan scan blocks execution if any
# tagged resources from a prior session exist.

USER_EMAIL = CALLER_USER

CLEANUP_MANIFEST = [
    CleanupResource(
        type="uc_registered_model",
        id=REGISTERED_MODEL_NAME,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP MODEL IF EXISTS {REGISTERED_MODEL_NAME}\"",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=ROUNDTRIP_TABLE_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {ROUNDTRIP_TABLE_FQN}\"",
    ),
    CleanupResource(
        type="mlflow_experiment",
        id=f"/Users/{CALLER_USER}/labrun-pyfunc-rag",
        region="databricks",
        cli_delete_command=(
            "databricks experiments delete-experiment "
            "$(databricks experiments get-by-name "
            f"--experiment-name /Users/{CALLER_USER}/labrun-pyfunc-rag "
            "--output JSON | jq -r .experiment.experiment_id)"
        ),
    ),
    CleanupResource(
        type="uc_schema",
        id=LAB_SCHEMA_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP SCHEMA IF EXISTS {LAB_SCHEMA_FQN} CASCADE\"",
    ),
]
MLFLOW_EXPERIMENT_NAME = f"/Users/{CALLER_USER}/labrun-pyfunc-rag"


class DatabricksCleanupAdapter:
    """Inline adapter for UC + MLflow + serving + vector search resources."""

    def delete_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype == "uc_managed_table":
            run_sql(f"DROP TABLE IF EXISTS {rid}")
        elif rtype == "uc_view":
            run_sql(f"DROP VIEW IF EXISTS {rid}")
        elif rtype == "uc_volume":
            run_sql(f"DROP VOLUME IF EXISTS {rid}")
        elif rtype == "uc_volume_contents":
            volume_path = "/Volumes/" + rid.replace(".", "/") + "/"
            try:
                listing = list(w.files.list_directory_contents(volume_path))
            except Exception:
                return
            for entry in listing:
                try:
                    if entry.is_directory:
                        w.files.delete_directory(entry.path)
                    else:
                        w.files.delete(entry.path)
                except Exception:
                    pass
        elif rtype == "uc_schema":
            run_sql(f"DROP SCHEMA IF EXISTS {rid} CASCADE")
        elif rtype == "uc_registered_model":
            try:
                import mlflow
                mlflow.set_registry_uri("databricks-uc")
                mlflow.MlflowClient().delete_registered_model(name=rid)
            except Exception:
                pass
        elif rtype == "mlflow_experiment":
            try:
                import mlflow
                exp = mlflow.get_experiment_by_name(rid)
                if exp is not None:
                    mlflow.delete_experiment(exp.experiment_id)
            except Exception:
                pass
        elif rtype == "model_serving_endpoint":
            try:
                w.serving_endpoints.delete(name=rid)
                deadline = time.time() + 600
                while time.time() < deadline:
                    try:
                        w.serving_endpoints.get(name=rid)
                    except Exception:
                        return
                    time.sleep(5)
            except Exception:
                pass
        elif rtype == "vector_search_index":
            try:
                w.vector_search_indexes.delete_index(index_name=rid)
            except Exception:
                pass
        elif rtype == "vector_search_endpoint":
            try:
                w.vector_search_endpoints.delete_endpoint(endpoint_name=rid)
            except Exception:
                pass
        else:
            raise ValueError(f"DatabricksCleanupAdapter: unknown resource type {rtype!r}")

    def describe_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype in ("uc_managed_table", "uc_view"):
            parts = rid.split(".")
            if len(parts) != 3:
                return False
            catalog, schema, table = parts
            try:
                sql = (
                    "SELECT 1 FROM system.information_schema.tables "
                    f"WHERE table_catalog = '{catalog}' AND table_schema = '{schema}' "
                    f"AND table_name = '{table}'"
                )
                return len(run_sql(sql)["rows"]) > 0
            except Exception:
                return False
        if rtype == "uc_volume":
            parts = rid.split(".")
            if len(parts) != 3:
                return False
            catalog, schema, volume = parts
            try:
                sql = (
                    "SELECT 1 FROM system.information_schema.volumes "
                    f"WHERE volume_catalog = '{catalog}' AND volume_schema = '{schema}' "
                    f"AND volume_name = '{volume}'"
                )
                return len(run_sql(sql)["rows"]) > 0
            except Exception:
                return False
        if rtype == "uc_volume_contents":
            volume_path = "/Volumes/" + rid.replace(".", "/") + "/"
            try:
                listing = list(w.files.list_directory_contents(volume_path))
                return len(listing) > 0
            except Exception:
                return False
        if rtype == "uc_schema":
            parts = rid.split(".")
            if len(parts) != 2:
                return False
            catalog, schema = parts
            try:
                sql = (
                    "SELECT 1 FROM system.information_schema.schemata "
                    f"WHERE catalog_name = '{catalog}' AND schema_name = '{schema}'"
                )
                return len(run_sql(sql)["rows"]) > 0
            except Exception:
                return False
        if rtype == "uc_registered_model":
            try:
                import mlflow
                mlflow.set_registry_uri("databricks-uc")
                mlflow.MlflowClient().get_registered_model(name=rid)
                return True
            except Exception:
                return False
        if rtype == "mlflow_experiment":
            try:
                import mlflow
                return mlflow.get_experiment_by_name(rid) is not None
            except Exception:
                return False
        if rtype == "model_serving_endpoint":
            try:
                w.serving_endpoints.get(name=rid)
                return True
            except Exception:
                return False
        if rtype == "vector_search_index":
            try:
                w.vector_search_indexes.get_index(index_name=rid)
                return True
            except Exception:
                return False
        if rtype == "vector_search_endpoint":
            try:
                w.vector_search_endpoints.get_endpoint(endpoint_name=rid)
                return True
            except Exception:
                return False
        return False

    def tag_scan(self, credentials, lab_slug, region):
        arns = []
        queries = [
            (
                "SELECT catalog_name, schema_name FROM system.information_schema.schema_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}",
            ),
            (
                "SELECT catalog_name, schema_name, table_name FROM system.information_schema.table_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}.{row[2]}",
            ),
        ]
        for sql, fmt in queries:
            try:
                result = run_sql(sql)
            except Exception:
                continue
            for row in result["rows"]:
                arns.append(fmt(row))
        return arns


CLEANUP_ADAPTER = DatabricksCleanupAdapter()


def _atexit_cleanup():
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans():
    return CLEANUP_ADAPTER.tag_scan(DATABRICKS_CREDENTIALS, LAB_TAG_VALUE, "databricks")


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing UC objects tagged {LAB_TAG_KEY}={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("Run the cleanup cell at the bottom first, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")


## Stand up the schema, MLflow experiment, and serializable chain code

`mlflow.pyfunc.PythonModel` loads inside a fresh Python process at predict time, so anything the chain depends on has to be importable from inside the model's environment. The cleanest way is to put the chain class in a single cell where every helper it needs is defined in the global scope, then point `log_model` at the same module via `code_paths` or rely on cloudpickle to capture the closure.

This cell creates the UC schema, points MLflow at the per-user experiment, and configures the UC registry as the model registry target.


In [ ]:
# NBVAL_SKIP
# Schema, experiment, and registry wiring.

import mlflow
import pandas as pd

run_sql(f"CREATE SCHEMA IF NOT EXISTS {LAB_SCHEMA_FQN}")
run_sql(f"ALTER SCHEMA {LAB_SCHEMA_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")  # three-level UC names route here
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
print(f"MLflow experiment ready: {MLFLOW_EXPERIMENT_NAME}")
print(f"Registry URI: databricks-uc (three-level names route to Unity Catalog)")


## Task 1: Define the RAGAgent pyfunc class

Subclass `mlflow.pyfunc.PythonModel`. Two methods carry the contract:

- `load_context(self, context)`: invoked once per process when the model is loaded back. Construct the `WorkspaceClient` here, not in `__init__`, because UC-registered models are loaded by Mosaic AI Model Serving in a fresh process where the constructor's state is gone.
- `predict(self, context, model_input)`: invoked per inference. The lab signature uses an array of strings as input, so `model_input` is a list. Apply pre-processing (`lower()`, `strip()`, reject if longer than 2000 chars), run the inline retrieve-and-answer chain, apply post-processing (truncate to 1500 chars, append a one-line citation footer).

The validator instantiates the class, calls `predict(None, ["  HELLO WORLD  "])` against a stub context, and asserts the response is a non-empty single-element list whose item is a string. It also intercepts the underlying chain to confirm pre-processing was actually applied before the chain was called.


In [ ]:
# NBVAL_SKIP
# Task 1: RAGAgent(mlflow.pyfunc.PythonModel) with pre and post processing.

from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

CORPUS = [
    ("doc-1", "Databricks Free Edition signs up via email OTP, Google, or Microsoft."),
    ("doc-2", "Mosaic AI Vector Search Delta-Sync indexes auto-update from a Delta table."),
    ("doc-3", "Foundation Model API pay-per-token endpoints bill cents per session."),
    ("doc-4", "Unity Catalog uses three-level naming: catalog.schema.table."),
]

MAX_INPUT_LEN = 2000
MAX_OUTPUT_LEN = 1500


class RAGAgent(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        # Imports inside load_context so a re-loaded model has a fresh import graph.
        from databricks.sdk import WorkspaceClient as WC
        self._wc = WC(host=DATABRICKS_CREDENTIALS["host"], token=DATABRICKS_CREDENTIALS["token"])

    def _retrieve(self, question):
        # Cheap inline retriever: pick the chunk with most overlapping lowercased tokens.
        q_tokens = set(question.lower().split())
        scored = []
        for doc_id, text in CORPUS:
            t_tokens = set(text.lower().split())
            scored.append((len(q_tokens & t_tokens), doc_id, text))
        scored.sort(reverse=True)
        return scored[0][1], scored[0][2]

    def _answer(self, question, context_text):
        resp = self._wc.serving_endpoints.query(
            name=LLM_ENDPOINT,
            messages=[
                ChatMessage(role=ChatMessageRole.SYSTEM,
                            content="Answer using only the provided context."),
                ChatMessage(role=ChatMessageRole.USER,
                            content=f"Context: {context_text}\n\nQuestion: {question}"),
            ],
            max_tokens=200,
        )
        return resp.choices[0].message.content or ""

    def predict(self, context, model_input, params=None):
        # YOUR CODE: coerce model_input into a list of strings (it can arrive as a list
        #            or a pandas DataFrame with one column named "query")
        # YOUR CODE: for each query: lowercase, strip; reject (raise ValueError)
        #            if longer than MAX_INPUT_LEN
        # YOUR CODE: call self._retrieve(cleaned) -> (doc_id, chunk_text)
        # YOUR CODE: call self._answer(cleaned, chunk_text) -> raw_response
        # YOUR CODE: truncate raw_response to MAX_OUTPUT_LEN
        # YOUR CODE: append a citation footer of the form
        #            "\n\nSources: doc-1" (use the retrieved doc_id)
        # YOUR CODE: return the list of finalized strings (one per input)
        return [""] * len(model_input)


In [ ]:
# NBVAL_SKIP
# Checkpoint 1: RAGAgent predict() applies pre-processing and returns the
# expected shape (list of strings, one per input).


def checkpoint_1(session):
    agent = RAGAgent()
    # Manually call load_context so the WorkspaceClient is wired up the same way
    # MLflow will do at serving time.
    try:
        agent.load_context(None)
    except Exception as exc:
        return CheckpointResult(
            status="error",
            error_reason=f"load_context raised: {exc!r}",
        )

    # Intercept _retrieve so we can confirm the pre-processed query was passed in.
    captured = {}
    original_retrieve = agent._retrieve
    def spy(q):
        captured["q"] = q
        return original_retrieve(q)
    agent._retrieve = spy

    try:
        out = agent.predict(None, ["  HELLO WORLD  "])
    except Exception as exc:
        return CheckpointResult(
            status="error",
            error_reason=f"predict() raised: {exc!r}",
        )

    if not isinstance(out, list) or len(out) != 1:
        return CheckpointResult(
            status="fail",
            error_reason=f"predict() returned {out!r}; expected list of one string.",
        )
    if not isinstance(out[0], str) or not out[0].strip():
        return CheckpointResult(
            status="fail",
            error_reason=f"predict() returned empty/non-string item: {out[0]!r}",
        )
    if captured.get("q") != "hello world":
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Pre-processing did not lowercase or strip; retriever saw {captured.get(\"q\")!r}; expected \"hello world\"."
            ),
        )
    if "Sources:" not in out[0]:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Post-processing did not append a citation footer; got {out[0]!r}."
            ),
        )
    return CheckpointResult(status="pass")


check(1, checkpoint_1)


<details><summary>Hint 1 (nudge)</summary>

Coerce the input shape first. If `model_input` arrived as a pandas DataFrame, use the first column; if it is already a list of strings, use it directly. Then apply your pre and post processing per query and return the list.

</details>

<details><summary>Hint 2 (direction)</summary>

The typical input-shape handler is two lines:

```
if hasattr(model_input, "iloc"):
    queries = list(model_input.iloc[:, 0])
else:
    queries = list(model_input)
```

Per query: `cleaned = query.lower().strip()`, raise `ValueError` if `len(cleaned) > MAX_INPUT_LEN`, retrieve, answer, truncate, append the footer.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1:

```python
def predict(self, context, model_input, params=None):
    if hasattr(model_input, "iloc"):
        queries = list(model_input.iloc[:, 0])
    else:
        queries = list(model_input)
    out = []
    for q in queries:
        cleaned = str(q).lower().strip()
        if len(cleaned) > MAX_INPUT_LEN:
            raise ValueError(f"Input length {len(cleaned)} exceeds {MAX_INPUT_LEN}")
        doc_id, chunk_text = self._retrieve(cleaned)
        raw = self._answer(cleaned, chunk_text)
        trimmed = raw[:MAX_OUTPUT_LEN]
        out.append(f"{trimmed}\n\nSources: {doc_id}")
    return out
```

</details>


**Wallet check.** One LLM call from the checkpoint instantiation (the predict() call on the fixture string). About 250 tokens at fractions of a cent. UC schema creation and the MLflow experiment touch are free.


## Task 2: Log the pyfunc with an explicit signature

The signature is the contract Model Serving and downstream consumers read. Build it via `mlflow.models.infer_signature(input_example, output_example)` and pass it to `log_model`. The lab's canonical input shape is an array of strings (chat models default), one query per element.

The `log_model` call lives inside an `mlflow.start_run()` so the artifacts land in the per-user experiment. Capture the resulting `model_info.model_uri` because Task 3 registers from that URI.


In [ ]:
# NBVAL_SKIP
# Task 2: log the pyfunc model with explicit signature.

from mlflow.models import infer_signature

input_example = ["How do I sign up for Free Edition?"]
output_example = ["Databricks Free Edition signs up via email OTP, Google, or Microsoft.\n\nSources: doc-1"]

# YOUR CODE: build signature = infer_signature(input_example, output_example)
# YOUR CODE: open mlflow.start_run() as run
# YOUR CODE:   model_info = mlflow.pyfunc.log_model(
# YOUR CODE:       artifact_path='rag_agent',
# YOUR CODE:       python_model=RAGAgent(),
# YOUR CODE:       signature=signature,
# YOUR CODE:       input_example=input_example,
# YOUR CODE:       pip_requirements=['mlflow==2.20.0', 'databricks-sdk==0.40.0'],
# YOUR CODE:   )
# YOUR CODE: store model_info.model_uri in MODEL_URI for the next task.
MODEL_URI = None


In [ ]:
# NBVAL_SKIP
# Checkpoint 2: a recent MLflow run in the experiment contains a logged
# pyfunc model with a signature whose input is an array of strings.


def checkpoint_2(session):
    import mlflow
    exp = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
    if exp is None:
        return CheckpointResult(
            status="fail",
            error_reason=f"MLflow experiment {MLFLOW_EXPERIMENT_NAME} not found.",
        )
    runs = mlflow.search_runs(
        experiment_ids=[exp.experiment_id],
        max_results=5,
        order_by=["attributes.start_time DESC"],
        output_format="list",
    )
    if not runs:
        return CheckpointResult(
            status="fail",
            error_reason="No MLflow runs found. Did mlflow.pyfunc.log_model() run inside a start_run()?",
        )
    client = mlflow.MlflowClient()
    for run in runs:
        try:
            mv = client.get_run(run.info.run_id)
            artifacts = client.list_artifacts(run.info.run_id)
        except Exception:
            continue
        artifact_names = {a.path for a in artifacts}
        if "rag_agent" not in artifact_names:
            continue
        # Load the MLmodel signature.
        try:
            model_info = mlflow.models.get_model_info(model_uri=f"runs:/{run.info.run_id}/rag_agent")
            sig = model_info.signature
        except Exception as exc:
            return CheckpointResult(
                status="fail",
                error_reason=f"Could not read signature for run {run.info.run_id}: {exc!r}",
            )
        if sig is None:
            return CheckpointResult(
                status="fail",
            error_reason=(
                f"Run {run.info.run_id} logged the pyfunc without a signature; "
                f"call infer_signature and pass signature=... to log_model."
            ),
        )
        input_repr = repr(sig.inputs).lower()
        if "string" not in input_repr:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Run {run.info.run_id} signature inputs do not include a string type. "
                    f"Inputs: {sig.inputs!r}"
                ),
            )
        return CheckpointResult(status="pass")
    return CheckpointResult(
        status="fail",
        error_reason="No recent run logged an rag_agent pyfunc artifact. Re-run Task 2.",
    )


check(2, checkpoint_2)


<details><summary>Hint 1 (nudge)</summary>

The error message tells you which piece is missing: the run does not exist (you forgot the `mlflow.start_run()`), the artifact is missing (your `log_model` call did not run), or the signature is missing (you forgot `signature=`). Read it before peeking further.

</details>

<details><summary>Hint 2 (direction)</summary>

`infer_signature(input_example, output_example)` is a single call. Then `mlflow.pyfunc.log_model(artifact_path="rag_agent", python_model=RAGAgent(), signature=signature, input_example=input_example, pip_requirements=[...])` inside a `with mlflow.start_run() as run:` block. Capture `model_info.model_uri` to a module-level constant so the next task can register from it.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2:

```python
signature = infer_signature(input_example, output_example)
with mlflow.start_run() as run:
    model_info = mlflow.pyfunc.log_model(
        artifact_path="rag_agent",
        python_model=RAGAgent(),
        signature=signature,
        input_example=input_example,
        pip_requirements=[
            "mlflow==2.20.0",
            "databricks-sdk==0.40.0",
        ],
    )
    MODEL_URI = model_info.model_uri
print(f"Logged: {MODEL_URI}")
```

</details>


**Wallet check.** Logging a pyfunc model is free on Databricks. The model artifact stores under the experiment's storage location which Databricks manages on Free Edition. The largest piece of cost in this lab is the round-trip inference in Task 4; everything before that is metastore writes plus an artifact upload.


## Task 3: Register to Unity Catalog and set the champion alias

Two SDK calls. First, `mlflow.register_model(model_uri=MODEL_URI, name=REGISTERED_MODEL_NAME)`. The three-level name routes the call to the UC registry (the registry URI was set to `databricks-uc` at setup). Wait for the version to reach status `READY` before setting the alias; aliases on still-creating versions return an error.

Second, `mlflow.MlflowClient().set_registered_model_alias(name=..., alias='champion', version=version)`. The version comes from the `register_model` return value.

The lab also tags the registered model with the labrun lab slug so the orphan scan can find it on a re-run.


In [ ]:
# NBVAL_SKIP
# Task 3: register the pyfunc to UC and set the champion alias.

client = mlflow.MlflowClient()

# YOUR CODE: result = mlflow.register_model(model_uri=MODEL_URI, name=REGISTERED_MODEL_NAME)
# YOUR CODE: poll client.get_model_version(name=REGISTERED_MODEL_NAME, version=result.version)
#            in a loop until status == 'READY' or a 120-second deadline
# YOUR CODE: client.set_registered_model_alias(
# YOUR CODE:     name=REGISTERED_MODEL_NAME, alias=CHAMPION_ALIAS, version=result.version)
# YOUR CODE: client.set_registered_model_tag(
# YOUR CODE:     name=REGISTERED_MODEL_NAME, key=LAB_TAG_KEY, value=LAB_TAG_VALUE)
# YOUR CODE: print the registered version number and the alias for the operator.


In [ ]:
# NBVAL_SKIP
# Checkpoint 3: UC-registered model exists at the three-level name with at
# least one version, and the champion alias resolves to a specific version.


def checkpoint_3(session):
    import mlflow
    client = mlflow.MlflowClient()
    try:
        rm = client.get_registered_model(name=REGISTERED_MODEL_NAME)
    except Exception as exc:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Registered model {REGISTERED_MODEL_NAME} not found. "
                f"Did you call mlflow.register_model with the three-level name? Error: {exc!r}"
            ),
        )
    if not rm.latest_versions:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"{REGISTERED_MODEL_NAME} has no versions yet. "
                f"The version may still be creating; re-run the cell."
            ),
        )
    try:
        mv = client.get_model_version_by_alias(name=REGISTERED_MODEL_NAME, alias=CHAMPION_ALIAS)
    except Exception as exc:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Alias {CHAMPION_ALIAS!r} not set on {REGISTERED_MODEL_NAME}. "
                f"Set it via client.set_registered_model_alias(...). Error: {exc!r}"
            ),
        )
    if not mv.version:
        return CheckpointResult(
            status="fail",
            error_reason=f"Alias {CHAMPION_ALIAS!r} did not resolve to a version.",
        )
    return CheckpointResult(status="pass")


check(3, checkpoint_3)


<details><summary>Hint 1 (nudge)</summary>

Two calls plus a wait. The wait is the trap: if you set the alias before the version is READY, the alias call returns "version not ready." A short polling loop with a 120-second cap solves it.

</details>

<details><summary>Hint 2 (direction)</summary>

`result = mlflow.register_model(model_uri=MODEL_URI, name=REGISTERED_MODEL_NAME)`. Then loop:

```
deadline = time.time() + 120
while time.time() < deadline:
    mv = client.get_model_version(name=REGISTERED_MODEL_NAME, version=result.version)
    if mv.status == "READY":
        break
    time.sleep(3)
```

then `client.set_registered_model_alias(...)`. Finally `client.set_registered_model_tag(...)` so the orphan scan can find the model on a re-run.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3:

```python
result = mlflow.register_model(model_uri=MODEL_URI, name=REGISTERED_MODEL_NAME)

deadline = time.time() + 120
while time.time() < deadline:
    mv = client.get_model_version(name=REGISTERED_MODEL_NAME, version=result.version)
    if mv.status == "READY":
        break
    time.sleep(3)
else:
    raise TimeoutError("Model version did not reach READY in 120s")

client.set_registered_model_alias(
    name=REGISTERED_MODEL_NAME, alias=CHAMPION_ALIAS, version=result.version
)
client.set_registered_model_tag(
    name=REGISTERED_MODEL_NAME, key=LAB_TAG_KEY, value=LAB_TAG_VALUE
)
print(f"Registered version {result.version} aliased as {CHAMPION_ALIAS}.")
```

</details>


**Wallet check.** Still free. UC model registration writes to the metastore and the experiment storage. The alias and tag calls are metastore touches. The round-trip inference in Task 4 is the first cell with non-zero token spend.


## Task 4: Load the model back via the alias URI and persist a round-trip result

Load the model via `mlflow.pyfunc.load_model(f"models:/{REGISTERED_MODEL_NAME}@{CHAMPION_ALIAS}")`. The `@<alias>` shorthand routes through the registry to whichever version the alias points at right now, which is exactly the production-deploy pattern.

Run a single fixture query through the loaded model, persist the result to `workspace.default.labrun_genai_pyfunc.roundtrip_results`, and let the checkpoint diff the row count and the response shape.


In [ ]:
# NBVAL_SKIP
# Task 4: load via alias URI and persist a round-trip inference result.

# Create the round-trip results table up front.
run_sql(
    f"CREATE OR REPLACE TABLE {ROUNDTRIP_TABLE_FQN} ("
    "query STRING, response STRING, model_version STRING)"
)
run_sql(f"ALTER TABLE {ROUNDTRIP_TABLE_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

# YOUR CODE: alias_uri = f"models:/{REGISTERED_MODEL_NAME}@{CHAMPION_ALIAS}"
# YOUR CODE: loaded = mlflow.pyfunc.load_model(alias_uri)
# YOUR CODE: fixture = "How do I sign up for Free Edition?"
# YOUR CODE: result = loaded.predict([fixture])  -> list of one string
# YOUR CODE: response = result[0]
# YOUR CODE: client = mlflow.MlflowClient()
# YOUR CODE: mv = client.get_model_version_by_alias(name=REGISTERED_MODEL_NAME, alias=CHAMPION_ALIAS)
# YOUR CODE: model_version = mv.version
# YOUR CODE: run_sql INSERT INTO ROUNDTRIP_TABLE_FQN VALUES (fixture, response, model_version)
# YOUR CODE:   (escape single quotes in the response with .replace(chr(39), chr(39)*2))


In [ ]:
# NBVAL_SKIP
# Checkpoint 4: round-trip results table has a row with non-empty response.


def checkpoint_4(session):
    try:
        rows = run_sql(
            f"SELECT query, response, model_version FROM {ROUNDTRIP_TABLE_FQN}"
        )["rows"]
    except Exception as exc:
        return CheckpointResult(
            status="error",
            error_reason=f"SELECT against {ROUNDTRIP_TABLE_FQN} failed: {exc!r}",
        )
    if not rows:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"{ROUNDTRIP_TABLE_FQN} has no rows. Did the round-trip insert run?"
            ),
        )
    for row in rows:
        query, response, version = row
        if not query or not response:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Row has empty query or response: ({query!r}, {response!r}). "
                    f"The load-back returned an empty inference; check the alias URI."
                ),
            )
        if not version:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Row {query!r} is missing the model_version. "
                    f"Resolve it via client.get_model_version_by_alias(...)."
                ),
            )
    return CheckpointResult(status="pass")


check(4, checkpoint_4)


<details><summary>Hint 1 (nudge)</summary>

`mlflow.pyfunc.load_model` takes the alias URI string. The result is a callable wrapper with `.predict(...)`. Run the fixture through it, save the response.

</details>

<details><summary>Hint 2 (direction)</summary>

```
alias_uri = f"models:/{REGISTERED_MODEL_NAME}@{CHAMPION_ALIAS}"
loaded = mlflow.pyfunc.load_model(alias_uri)
response = loaded.predict(["How do I sign up for Free Edition?"])[0]
```

Then resolve the version via the client (`get_model_version_by_alias`) and write a single INSERT with the query, response, and version.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 4:

```python
alias_uri = f"models:/{REGISTERED_MODEL_NAME}@{CHAMPION_ALIAS}"
loaded = mlflow.pyfunc.load_model(alias_uri)
fixture = "How do I sign up for Free Edition?"
response = loaded.predict([fixture])[0]

client = mlflow.MlflowClient()
mv = client.get_model_version_by_alias(name=REGISTERED_MODEL_NAME, alias=CHAMPION_ALIAS)
safe_q = fixture.replace(chr(39), chr(39) * 2)
safe_r = response.replace(chr(39), chr(39) * 2)
run_sql(
    f"INSERT INTO {ROUNDTRIP_TABLE_FQN} VALUES "
    f"('{safe_q}', '{safe_r}', '{mv.version}')"
)
print(f"Round-trip persisted. Version: {mv.version}")
print(response[:200])
```

</details>


**Wallet check.** One full chain invocation through the loaded model: one LLM call at about 300 tokens. Total session spend rounds to a cent. UC model storage on Free Edition is included.


## Cleanup

Tear down the round-trip table, the UC-registered model (cascading every version), the MLflow experiment, and the lab schema. Idempotent on a second run.


In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation
# order using the inline Databricks adapter. Prints the canonical summary
# from RESOURCE-SAFETY-SPEC Rule 6 and sys.exit(1) on any failure.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Databricks workspace may still hold lab objects. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)


**Session total: about $0.02.** One LLM call from the checkpoint plus one from the load-back, both at the 70B endpoint, both fractions of a cent in token spend. MLflow logging and UC model registration are free on Free Edition. The cleanup cell tore down the round-trip table, dropped the registered model with every version, deleted the per-user experiment, and dropped the schema.


## Reflection

These are not graded. They are for you.

1. Walk through what `mlflow.pyfunc.PythonModel.load_context()` does that `__init__` does not. Why does chain construction belong inside `load_context()` for a UC-registered model that will later be loaded by a Mosaic AI Model Serving endpoint, rather than inside the constructor?

2. Aliases (champion, challenger) replaced stages (Staging, Production) as the canonical promotion mechanism in UC-registered models. What is the operational difference between updating an alias and updating a stage, and why does the alias model fit a multi-environment rollout pattern better?

## Exam-Style Practice

**Question 1 (Domain 4, basic elements of a RAG application):**

A GenAI engineer is registering a RAG application to Unity Catalog using MLflow. The exam guide lists the basic elements they must specify when logging the model: model flavor, embedding model, retriever, dependencies, input examples, model signature. Which element does `mlflow.models.infer_signature(input_example, output_example)` produce?

A. The model flavor (e.g., `pyfunc`).
B. The dependencies (pip requirements).
C. The model signature (input/output schema).
D. The embedding model reference.

<details><summary>Show answer</summary>

**Correct: C.**

- A is wrong: the model flavor is set by which `log_model` call you use (`mlflow.pyfunc.log_model` vs `mlflow.langchain.log_model`), not by `infer_signature`.
- B is wrong: dependencies come from `pip_requirements`, `conda_env`, or the auto-detected environment.
- C is correct: `infer_signature` produces the input/output schema (an `mlflow.models.ModelSignature`) from example input and output values. It is the canonical way to attach a schema at log time.
- D is wrong: the embedding model is referenced inside the chain code itself, not produced by `infer_signature`.

</details>

**Question 2 (Domain 4, UC model registration vs workspace registry):**

A GenAI engineer is registering a RAG model. The platform team requires the registration target be Unity Catalog (not the workspace MLflow registry). Which call shape is correct?

A. `mlflow.register_model(model_uri='runs:/<run_id>/model', name='rag_agent')`.
B. `mlflow.register_model(model_uri='runs:/<run_id>/model', name='workspace.default.labrun_genai_pyfunc.rag_agent')`.
C. `mlflow.register_model(model_uri='runs:/<run_id>/model', name='Production/rag_agent')`.
D. `mlflow.register_model(model_uri='runs:/<run_id>/model', name='workspace.default.rag_agent')` (no schema).

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: a single-token name registers to the workspace MLflow registry, not UC.
- B is correct: the three-level `catalog.schema.model` name routes registration to Unity Catalog. UC requires three levels exactly.
- C is wrong: `Production/` is workspace-registry stage syntax, not UC.
- D is wrong: a two-level name is invalid for UC registration. The three levels are catalog, schema, model.

</details>
